# Enova Technologies - Synthetic Data Generation

In [ ]:
"""
Enova Technologies - Synthetic Data Generation
===============================================
Generates nine raw CSV files for the Talent Intelligence Analytics project.
The data covers 18 months of company growth from 50 to 300 employees
following a Series B funding round.

Output files saved to Google Drive:
  data/raw/talentflow_jobs.csv
  data/raw/talentflow_candidates.csv
  data/raw/talentflow_applications.csv
  data/raw/talentflow_pipeline_events.csv
  data/raw/talentflow_offers.csv
  data/raw/peoplecore_employees.csv
  data/raw/peoplecore_performance.csv
  data/raw/peoplecore_compensation.csv
  data/raw/finance_headcount_plan.csv

"""

---
## Cell 0: COLAB SETUP

In [1]:
# =============================================================
# CELL 0: COLAB SETUP
# =============================================================
# Installs Faker for name generation, mounts Google Drive,
# and creates the folder structure. Google will prompt for
# Drive authorisation when drive.mount() runs.
# =============================================================

!pip install faker --quiet

from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/enova-talent-intelligence'

PATHS = {
    'raw':     f'{BASE}/data/raw',
    'clean':   f'{BASE}/data/clean',
    'outputs': f'{BASE}/outputs',
    'docs':    f'{BASE}/docs',
    'src':     f'{BASE}/src/01_data_generation',
}

for p in PATHS.values():
    os.makedirs(p, exist_ok=True)

print('Folders ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.0 MB/s eta 0:00:00
Mounted at /content/drive
Folders ready.


---
## Cell 1: IMPORTS AND RANDOM SEED

In [2]:
# =============================================================
# CELL 1: IMPORTS AND RANDOM SEED
# =============================================================
# Seed 42 makes results reproducible across runs. Without a
# fixed seed the data changes each time, which would break any
# analysis written against a specific dataset.
# =============================================================

import random
import numpy as np
import pandas as pd
from faker import Faker
from datetime import date, timedelta
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# One Faker instance per nationality. Enova is headquartered in
# Vilnius with employees across Eastern and Western Europe, so
# names need to reflect that mix.
fake_lt      = Faker('lt_LT')
fake_pl      = Faker('pl_PL')
fake_de      = Faker('de_DE')
fake_gb      = Faker('en_GB')
fake_generic = Faker('en_US')

for f in [fake_lt, fake_pl, fake_de, fake_gb, fake_generic]:
    f.seed_instance(RANDOM_SEED)

print('Libraries loaded. Seed set to 42.')

Libraries loaded. Seed set to 42.


---
## Cell 2: CONFIGURATION

In [3]:
# =============================================================
# CELL 2: CONFIGURATION
# =============================================================
# All volume targets, date boundaries, and distributions live
# here. Changing a value in this cell propagates throughout
# the rest of the notebook without touching generation logic.
# =============================================================

# Date boundaries
# TODAY is fixed at 2025-07-01 rather than the actual current
# date so the dataset stays consistent over time.
TODAY             = date(2025, 7, 1)
TALENTFLOW_GOLIVE = date(2024, 1, 1)   # When TalentFlow was implemented
COMPANY_FOUNDED   = date(2023, 1, 1)

# Volume targets
N_JOBS             = 45
N_APPLICATIONS     = 4200
N_ATS_HIRES        = 180   # Hires tracked through TalentFlow
N_OFFERS           = 200   # 180 accepted + 20 declined = 90% acceptance rate
N_ACTIVE_EMPLOYEES = 300
N_TERMINATED       = 45
N_PERF_REVIEWS     = 280
N_COMP_CHANGES     = 120

# TalentFlow uses these department names. PeopleCore uses
# different names for the same departments. That mismatch is
# data quality issue DQ-HRIS-02, injected deliberately in Cell 13.
DEPARTMENTS = [
    'Technology',
    'Product',
    'Data and Analytics',
    'Commercial',
    'Finance',
    'Operations',
    'People',
    'Legal and Compliance',
]

# Technology gets the most requisitions because it is the
# hardest department to hire for and the largest team.
DEPT_JOB_COUNTS = {
    'Technology':           15,
    'Product':               6,
    'Data and Analytics':    5,
    'Commercial':            7,
    'Finance':               3,
    'Operations':            4,
    'People':                3,
    'Legal and Compliance':  2,
}   # Total = 45

JOB_LEVELS = ['ic1', 'ic2', 'ic3', 'manager', 'senior_manager', 'director', 'vp']

# Canonical stage names. Cell 13 replaces about 15% of these
# with messy variants to simulate real-world ATS export quality.
PIPELINE_STAGES_ORDERED = [
    'Applied',
    'Resume Screen',
    'Recruiter Screen',
    'Hiring Manager Screen',
    'Technical Assessment',
    'Technical Interview',
    'Final Panel Interview',
    'Reference Check',
    'Offer',
    'Offer Accepted',
    'Hired',
]

# Employee Referral is set to 11% to match Key Finding 2:
# referrals are 11% of applications but 28% of hires.
SOURCES = [
    'LinkedIn Organic',
    'LinkedIn Paid',
    'LinkedIn Recruiter',
    'Employee Referral',
    'Direct Application',
    'Indeed',
    'Glassdoor',
    'Recruitment Agency',
    'GitHub Sourcing',
    'University Partnership',
    'Other',
]
SOURCE_WEIGHTS = [0.28, 0.06, 0.18, 0.11, 0.12, 0.08, 0.06, 0.05, 0.02, 0.02, 0.02]

SOURCE_SUBTYPES = {
    'LinkedIn Organic':       'Job Posting',
    'LinkedIn Paid':          'Sponsored Job',
    'LinkedIn Recruiter':     'InMail Outreach',
    'Employee Referral':      'Internal Referral',
    'Direct Application':     'Careers Page',
    'Indeed':                 'Indeed Apply',
    'Glassdoor':              'Glassdoor Apply',
    'Recruitment Agency':     'Agency Submission',
    'GitHub Sourcing':        'GitHub Profile',
    'University Partnership': 'Careers Fair',
    'Other':                  'Unknown',
}

EMPLOYEE_LOCATIONS = [
    ('Vilnius',   'Lithuania',       'EUR'),
    ('Kaunas',    'Lithuania',       'EUR'),
    ('Warsaw',    'Poland',          'EUR'),
    ('Krakow',    'Poland',          'EUR'),
    ('Berlin',    'Germany',         'EUR'),
    ('Amsterdam', 'Netherlands',     'EUR'),
    ('London',    'United Kingdom',  'GBP'),
    ('Tallinn',   'Estonia',         'EUR'),
    ('Riga',      'Latvia',          'EUR'),
    ('Kyiv',      'Ukraine',         'EUR'),
    ('Remote',    'Lithuania',       'EUR'),
]
LOCATION_WEIGHTS = [0.30, 0.10, 0.12, 0.08, 0.08, 0.07, 0.07, 0.06, 0.04, 0.04, 0.04]

# Annual salary ranges by country and level.
# The boolean at the end of each tuple flags whether employees
# at that country/level are candidates for the salary type
# data quality issue (DQ-HRIS-01): monthly amounts entered as annual.
SALARY_RANGES = {
    ('Lithuania', 'ic1'):          (22000,  34000,  'EUR', True),
    ('Lithuania', 'ic2'):          (34000,  52000,  'EUR', True),
    ('Lithuania', 'ic3'):          (52000,  78000,  'EUR', True),
    ('Lithuania', 'manager'):      (68000,  95000,  'EUR', False),
    ('Lithuania', 'senior_manager'): (85000, 115000, 'EUR', False),
    ('Lithuania', 'director'):     (105000, 145000, 'EUR', False),
    ('Lithuania', 'vp'):           (135000, 185000, 'EUR', False),
    ('Poland', 'ic1'):             (25000,  38000,  'EUR', True),
    ('Poland', 'ic2'):             (38000,  58000,  'EUR', True),
    ('Poland', 'ic3'):             (58000,  85000,  'EUR', True),
    ('Poland', 'manager'):         (72000, 100000,  'EUR', True),
    ('Poland', 'senior_manager'):  (88000, 118000,  'EUR', False),
    ('Poland', 'director'):        (108000, 148000, 'EUR', False),
    ('Poland', 'vp'):              (138000, 188000, 'EUR', False),
    ('Germany', 'ic1'):            (46000,  62000,  'EUR', False),
    ('Germany', 'ic2'):            (62000,  82000,  'EUR', False),
    ('Germany', 'ic3'):            (82000, 112000,  'EUR', False),
    ('Germany', 'manager'):        (92000, 128000,  'EUR', False),
    ('Germany', 'senior_manager'): (110000, 145000, 'EUR', False),
    ('Germany', 'director'):       (130000, 170000, 'EUR', False),
    ('Germany', 'vp'):             (160000, 210000, 'EUR', False),
    ('Netherlands', 'ic1'):        (44000,  60000,  'EUR', False),
    ('Netherlands', 'ic2'):        (60000,  80000,  'EUR', False),
    ('Netherlands', 'ic3'):        (80000, 108000,  'EUR', False),
    ('Netherlands', 'manager'):    (90000, 125000,  'EUR', False),
    ('Netherlands', 'senior_manager'): (108000, 142000, 'EUR', False),
    ('Netherlands', 'director'):   (128000, 168000, 'EUR', False),
    ('Netherlands', 'vp'):         (158000, 208000, 'EUR', False),
    ('United Kingdom', 'ic1'):     (36000,  52000,  'GBP', False),
    ('United Kingdom', 'ic2'):     (52000,  72000,  'GBP', False),
    ('United Kingdom', 'ic3'):     (72000,  98000,  'GBP', False),
    ('United Kingdom', 'manager'): (88000, 120000,  'GBP', False),
    ('United Kingdom', 'senior_manager'): (105000, 140000, 'GBP', False),
    ('United Kingdom', 'director'): (125000, 165000, 'GBP', False),
    ('United Kingdom', 'vp'):      (155000, 205000, 'GBP', False),
    ('Estonia', 'ic1'):            (24000,  36000,  'EUR', True),
    ('Estonia', 'ic2'):            (36000,  56000,  'EUR', True),
    ('Estonia', 'ic3'):            (56000,  80000,  'EUR', True),
    ('Estonia', 'manager'):        (68000,  95000,  'EUR', False),
    ('Estonia', 'senior_manager'): (85000, 115000,  'EUR', False),
    ('Estonia', 'director'):       (105000, 140000, 'EUR', False),
    ('Estonia', 'vp'):             (130000, 175000, 'EUR', False),
    ('Latvia', 'ic1'):             (20000,  30000,  'EUR', True),
    ('Latvia', 'ic2'):             (30000,  48000,  'EUR', True),
    ('Latvia', 'ic3'):             (48000,  72000,  'EUR', True),
    ('Latvia', 'manager'):         (62000,  88000,  'EUR', True),
    ('Latvia', 'senior_manager'):  (80000, 108000,  'EUR', False),
    ('Latvia', 'director'):        (100000, 135000, 'EUR', False),
    ('Latvia', 'vp'):              (125000, 168000, 'EUR', False),
    ('Ukraine', 'ic1'):            (18000,  28000,  'EUR', True),
    ('Ukraine', 'ic2'):            (28000,  44000,  'EUR', True),
    ('Ukraine', 'ic3'):            (44000,  68000,  'EUR', True),
    ('Ukraine', 'manager'):        (58000,  82000,  'EUR', True),
    ('Ukraine', 'senior_manager'): (75000, 100000,  'EUR', False),
    ('Ukraine', 'director'):       (92000, 125000,  'EUR', False),
    ('Ukraine', 'vp'):             (118000, 158000, 'EUR', False),
}

print('Configuration loaded.')
print(f'  Data window: {COMPANY_FOUNDED} to {TODAY}')
print(f'  TalentFlow live since: {TALENTFLOW_GOLIVE}')
print(f'  Jobs: {N_JOBS} | Applications: {N_APPLICATIONS} | Hires: {N_ATS_HIRES}')

Configuration loaded.
  Data window: 2023-01-01 to 2025-07-01
  TalentFlow live since: 2024-01-01
  Jobs: 45 | Applications: 4200 | Hires: 180


---
## Cell 3: REFERENCE DATA

In [4]:
# =============================================================
# CELL 3: REFERENCE DATA
# =============================================================
# Name pools, company names, job titles, and rejection reasons.
# Using real Eastern European tech company names makes the
# dataset look credible in a portfolio demo context.
# =============================================================

NAMES = {
    'Lithuanian_male':  ['Tomas', 'Mantas', 'Lukas', 'Jonas', 'Matas', 'Arnas',
                         'Domantas', 'Edvinas', 'Gintaras', 'Mindaugas', 'Rytis',
                         'Saulius', 'Tadas', 'Vilius', 'Erikas', 'Rokas', 'Dovydas'],
    'Lithuanian_female':['Inga', 'Ruta', 'Viktorija', 'Monika', 'Agne', 'Greta',
                         'Jolanta', 'Kristina', 'Laura', 'Neringa', 'Simona',
                         'Vaida', 'Egle', 'Aiste', 'Gabija', 'Lina'],
    'Polish_male':      ['Jakub', 'Mateusz', 'Piotr', 'Michal', 'Bartosz', 'Kamil',
                         'Marcin', 'Krzysztof', 'Tomasz', 'Lukasz', 'Adam', 'Filip',
                         'Damian', 'Rafal', 'Pawel', 'Maciej'],
    'Polish_female':    ['Anna', 'Marta', 'Karolina', 'Agnieszka', 'Katarzyna',
                         'Monika', 'Natalia', 'Paulina', 'Aleksandra', 'Weronika',
                         'Magdalena', 'Joanna', 'Ewelina'],
    'Ukrainian_male':   ['Oleksiy', 'Dmytro', 'Andriy', 'Mykola', 'Ivan', 'Vasyl',
                         'Serhiy', 'Volodymyr', 'Taras', 'Bohdan', 'Yaroslav',
                         'Artem', 'Viktor', 'Mykhailo', 'Oleh'],
    'Ukrainian_female': ['Olena', 'Natalia', 'Oksana', 'Iryna', 'Yulia', 'Tetyana',
                         'Kateryna', 'Svitlana', 'Daryna', 'Alina', 'Mariya',
                         'Halyna', 'Larysa'],
    'German_male':      ['Lukas', 'Felix', 'Max', 'Jonas', 'Leon', 'Julian', 'Tim',
                         'Alexander', 'David', 'Florian', 'Tobias', 'Christian',
                         'Stefan', 'Michael', 'Andreas', 'Philipp'],
    'German_female':    ['Julia', 'Anna', 'Laura', 'Sarah', 'Sophie', 'Lisa',
                         'Hannah', 'Lena', 'Lea', 'Jana', 'Katharina', 'Nicole',
                         'Sandra', 'Claudia'],
    'British_male':     ['James', 'Oliver', 'Jack', 'Harry', 'George', 'Charlie',
                         'Thomas', 'Oscar', 'William', 'Noah', 'Liam', 'Ethan',
                         'Daniel', 'Matthew', 'Andrew'],
    'British_female':   ['Emily', 'Sophie', 'Olivia', 'Jessica', 'Amy', 'Emma',
                         'Charlotte', 'Lucy', 'Hannah', 'Mia', 'Chloe', 'Grace',
                         'Lily', 'Isabelle', 'Ella'],
    'Other_male':       ['Aleksander', 'Mikhail', 'Andrei', 'Nikita', 'Vladislav',
                         'Rafal', 'Marius', 'Tiberiu', 'Bence', 'Janos'],
    'Other_female':     ['Elena', 'Anastasia', 'Daria', 'Oksana', 'Mirela',
                         'Zsuzsanna', 'Petra', 'Kinga', 'Ioana'],
}

LAST_NAMES = {
    'Lithuanian': ['Kazlauskas', 'Petrauskas', 'Jankauskas', 'Paulauskas',
                   'Stonkus', 'Butkus', 'Gudas', 'Levinskas', 'Mockus',
                   'Rimkus', 'Stankus', 'Urbonas', 'Zukauskas', 'Ambrazas',
                   'Daunoras', 'Grigas', 'Kairys', 'Noreika', 'Radzevicius'],
    'Polish':     ['Kowalski', 'Nowak', 'Wisniewski', 'Wojcik', 'Kowalczyk',
                   'Lewandowski', 'Zielinski', 'Szymanski', 'Wozniak',
                   'Kaminski', 'Dabrowska', 'Kozlowski', 'Jankowski', 'Mazur'],
    'Ukrainian':  ['Kovalenko', 'Melnyk', 'Shevchenko', 'Bondarenko',
                   'Tkachenko', 'Kravchenko', 'Kovalchuk', 'Oliynyk',
                   'Shevchuk', 'Moroz', 'Boyko', 'Lytovchenko', 'Marchenko'],
    'German':     ['Muller', 'Schmidt', 'Schneider', 'Fischer', 'Weber',
                   'Meyer', 'Wagner', 'Becker', 'Schulz', 'Hoffmann',
                   'Richter', 'Bauer', 'Koch', 'Hartmann', 'Klein'],
    'British':    ['Smith', 'Jones', 'Williams', 'Brown', 'Taylor', 'Davies',
                   'Evans', 'Wilson', 'Thomas', 'Roberts', 'Hughes', 'Lewis',
                   'Walker', 'Robinson', 'Clark'],
    'Other':      ['Popescu', 'Ionescu', 'Nagy', 'Toth', 'Kovacs',
                   'Ivanov', 'Petrov', 'Smirnov', 'Kuznetsov', 'Fedorov'],
}

# Companies representing the main talent pools Enova hires from.
# Gaming and Baltic tech companies are weighted heavily.
COMPANIES = [
    'Wargaming', 'Playtika', 'King', 'Vinted', 'Bolt', 'Pipedrive',
    'Nord Security', 'TransferGo', 'Adform', 'Tesonet',
    'Revolut', 'Wise', 'Zalando', 'Delivery Hero', 'N26',
    'Contentful', 'SumUp', 'Personio',
    'Devbridge', 'EPAM Systems', 'Accenture', 'CGI', 'Atea',
    'Danske Bank', 'SEB', 'Swedbank', 'Luminor',
    'Telia', 'Bite', 'LMT',
    'Cognizant', 'DXC Technology', 'Barclays Technology',
    'TrueLayer', 'Checkout.com', 'Paddle',
    'Ubisoft', 'EA Games', 'Zynga', 'Miniclip', 'Jam City',
    'Freelance', None,
]

JOB_TITLES = {
    'Technology': {
        'ic1': ['Junior Backend Engineer', 'Junior Frontend Engineer',
                'Junior QA Engineer', 'Junior DevOps Engineer'],
        'ic2': ['Backend Engineer', 'Frontend Engineer', 'Mobile Engineer',
                'DevOps Engineer', 'QA Engineer', 'Platform Engineer'],
        'ic3': ['Senior Backend Engineer', 'Senior Frontend Engineer',
                'Senior Mobile Engineer', 'Senior DevOps Engineer',
                'Security Engineer', 'Staff Engineer'],
        'manager':        ['Engineering Manager'],
        'senior_manager': ['Senior Engineering Manager'],
        'director':       ['Head of Engineering', 'Director of Engineering'],
        'vp':             ['VP of Engineering', 'CTO'],
    },
    'Product': {
        'ic1': ['Junior Product Designer', 'Associate Product Manager'],
        'ic2': ['Product Manager', 'Product Designer', 'UX Researcher'],
        'ic3': ['Senior Product Manager', 'Senior Product Designer',
                'Lead UX Researcher', 'Principal Product Manager'],
        'manager':        ['Product Lead', 'Group Product Manager'],
        'senior_manager': ['Senior Product Lead'],
        'director':       ['Head of Product', 'Director of Product'],
        'vp':             ['VP of Product', 'Chief Product Officer'],
    },
    'Data and Analytics': {
        'ic1': ['Junior Data Analyst', 'Junior BI Analyst'],
        'ic2': ['Data Analyst', 'Data Engineer', 'BI Analyst', 'Analytics Engineer'],
        'ic3': ['Senior Data Analyst', 'Senior Data Engineer',
                'Data Scientist', 'Senior Analytics Engineer'],
        'manager':        ['Data Engineering Manager', 'Analytics Manager'],
        'senior_manager': ['Senior Analytics Manager'],
        'director':       ['Head of Data', 'Director of Data and Analytics'],
        'vp':             ['VP of Data'],
    },
    'Commercial': {
        'ic1': ['Marketing Coordinator', 'Growth Analyst'],
        'ic2': ['Growth Manager', 'Performance Marketing Manager',
                'CRM Manager', 'SEO Specialist', 'Brand Manager'],
        'ic3': ['Senior Growth Manager', 'Senior Performance Marketer',
                'Senior CRM Manager'],
        'manager':        ['Marketing Manager', 'Growth Lead'],
        'senior_manager': ['Senior Marketing Manager'],
        'director':       ['Head of Growth', 'Head of Marketing'],
        'vp':             ['VP of Marketing', 'Chief Marketing Officer'],
    },
    'Finance': {
        'ic1': ['Finance Analyst', 'Accounts Assistant'],
        'ic2': ['Financial Analyst', 'FP&A Analyst', 'Management Accountant'],
        'ic3': ['Senior Financial Analyst', 'Senior FP&A Analyst'],
        'manager':        ['Finance Manager', 'FP&A Manager'],
        'senior_manager': ['Senior Finance Manager'],
        'director':       ['Head of Finance', 'Finance Director'],
        'vp':             ['VP of Finance', 'CFO'],
    },
    'Operations': {
        'ic1': ['Customer Support Specialist', 'Operations Analyst',
                'Trust and Safety Analyst'],
        'ic2': ['Senior Customer Support Specialist', 'Operations Specialist',
                'Partner Operations Manager', 'Fraud Analyst'],
        'ic3': ['Customer Support Lead', 'Senior Operations Specialist',
                'Trust and Safety Lead'],
        'manager':        ['Operations Manager', 'Customer Support Manager'],
        'senior_manager': ['Senior Operations Manager'],
        'director':       ['Head of Operations', 'Director of Operations'],
        'vp':             ['VP of Operations', 'COO'],
    },
    'People': {
        'ic1': ['People Operations Coordinator', 'TA Coordinator'],
        'ic2': ['HR Business Partner', 'Talent Acquisition Specialist',
                'L&D Specialist', 'Compensation and Benefits Analyst'],
        'ic3': ['Senior HR Business Partner', 'Senior TA Specialist'],
        'manager':        ['HR Manager', 'TA Manager'],
        'senior_manager': ['Senior HR Manager'],
        'director':       ['Head of People', 'Head of Talent Acquisition'],
        'vp':             ['VP of People', 'Chief People Officer'],
    },
    'Legal and Compliance': {
        'ic1': ['Legal Trainee', 'Compliance Analyst'],
        'ic2': ['Legal Counsel', 'Compliance Specialist'],
        'ic3': ['Senior Legal Counsel', 'Senior Compliance Manager'],
        'manager':        ['Legal Manager', 'Compliance Manager'],
        'senior_manager': ['Senior Legal Manager'],
        'director':       ['Head of Legal', 'Head of Compliance'],
        'vp':             ['VP of Legal', 'General Counsel'],
    },
}

REJECTION_REASONS = {
    'Resume Screen': [
        'Not enough experience',
        'Skills mismatch',
        'Location / visa mismatch',
        'Salary expectation out of range',
        'Role filled internally',
    ],
    'Recruiter Screen': [
        'Not enough experience',
        'Salary expectation out of range',
        'Availability too late',
        'Skills mismatch',
        'Preferred another candidate',
    ],
    'Hiring Manager Screen': [
        'Technical skills below bar',
        'Preferred another candidate',
        'Culture fit concern',
        'Communication skills',
        'Role requirements changed',
    ],
    'Technical Assessment': [
        'Technical skills below bar',
        'Assessment quality below bar',
        'Preferred another candidate',
    ],
    'Technical Interview': [
        'Technical skills below bar',
        'Problem solving below bar',
        'Preferred another candidate',
        'Culture fit concern',
    ],
    'Final Panel Interview': [
        'Preferred another candidate',
        'Culture fit concern',
        'Communication skills',
        'Technical skills below bar',
        'Role requirements changed',
    ],
    'Reference Check': [
        'Reference check raised concerns',
        'Offer rescinded - reference check',
    ],
}

WITHDRAWAL_REASONS = [
    'Accepted another offer',
    'No response from candidate',
    'Personal reasons',
    'Role no longer of interest',
]

OFFER_DECLINE_REASONS = [
    'compensation',
    'competing_offer',
    'location',
    'role_fit',
    'personal_reasons',
    'no_response',
]

print('Reference data loaded.')

Reference data loaded.


---
## Cell 4: HELPER FUNCTIONS

In [5]:
# =============================================================
# CELL 4: HELPER FUNCTIONS
# =============================================================
# Utility functions used throughout the notebook.
# The two most important ones are get_tech_assess_duration
# and get_stage_durations, which embed Key Findings 1 and 2
# directly into the generated timeline data.
# =============================================================

def rand_date(start: date, end: date) -> date:
    delta = (end - start).days
    if delta <= 0:
        return start
    return start + timedelta(days=random.randint(0, delta))


def days_later(d: date, min_days: int, max_days: int) -> date:
    return d + timedelta(days=random.randint(min_days, max_days))


def to_iso(d) -> str:
    if d is None:
        return None
    return d.strftime('%Y-%m-%d')


def get_salary(country: str, level: str) -> tuple:
    key = (country, level)
    if key not in SALARY_RANGES:
        key = ('Lithuania', level if level in SALARY_RANGES.get(
            'Lithuania', {}) else 'ic2')
        key = ('Lithuania', 'ic2')
    sal_min, sal_max, currency, _ = SALARY_RANGES[key]
    amount = round(random.randint(sal_min, sal_max) / 500) * 500
    return amount, currency


def get_name(nationality: str) -> tuple:
    gender_roll = random.random()
    if gender_roll < 0.49:
        gender, gender_key = 'm', 'male'
    elif gender_roll < 0.98:
        gender, gender_key = 'f', 'female'
    elif gender_roll < 0.99:
        gender, gender_key = 'non_binary', random.choice(['male', 'female'])
    else:
        gender, gender_key = 'prefer_not_to_say', random.choice(['male', 'female'])

    nat_map = {
        'Lithuania':      'Lithuanian',
        'Poland':         'Polish',
        'Ukraine':        'Ukrainian',
        'Germany':        'German',
        'United Kingdom': 'British',
    }
    pool_key = nat_map.get(nationality, 'Other')
    name_key = f'{pool_key}_{gender_key}'
    if name_key not in NAMES:
        name_key = f'Other_{gender_key}'

    first = random.choice(NAMES[name_key])
    last  = random.choice(LAST_NAMES.get(pool_key, LAST_NAMES['Other']))
    return first, last, gender


def get_job_level_for_dept(dept: str) -> str:
    weights = {
        'Technology':           [0.05, 0.35, 0.35, 0.12, 0.07, 0.05, 0.01],
        'Product':              [0.02, 0.30, 0.35, 0.15, 0.10, 0.07, 0.01],
        'Data and Analytics':   [0.05, 0.35, 0.35, 0.12, 0.07, 0.05, 0.01],
        'Commercial':           [0.05, 0.35, 0.30, 0.15, 0.08, 0.06, 0.01],
        'Finance':              [0.05, 0.40, 0.30, 0.15, 0.05, 0.05, 0.00],
        'Operations':           [0.20, 0.35, 0.25, 0.12, 0.05, 0.03, 0.00],
        'People':               [0.05, 0.40, 0.30, 0.15, 0.05, 0.05, 0.00],
        'Legal and Compliance': [0.00, 0.20, 0.40, 0.20, 0.10, 0.10, 0.00],
    }
    w = weights.get(dept, [0.05, 0.35, 0.35, 0.12, 0.07, 0.05, 0.01])
    return random.choices(JOB_LEVELS, weights=w, k=1)[0]


def get_stage_sequence(dept: str, level: str) -> list:
    # Defines which pipeline stages apply to each role type.
    # Not all roles go through all stages.
    base = ['Applied', 'Resume Screen', 'Recruiter Screen', 'Hiring Manager Screen']

    has_tech_assess   = dept in ['Technology', 'Data and Analytics', 'Product']
    has_tech_interview = (dept in ['Technology', 'Data and Analytics']
                          and level in ['ic3', 'manager', 'senior_manager',
                                        'director', 'vp'])
    has_ref_check = level in ['ic3', 'manager', 'senior_manager', 'director', 'vp']

    middle = []
    if has_tech_assess:
        middle.append('Technical Assessment')
    if has_tech_interview:
        middle.append('Technical Interview')
    middle.append('Final Panel Interview')
    if has_ref_check:
        middle.append('Reference Check')

    return base + middle + ['Offer']


def get_tech_assess_duration(dept: str) -> int:
    # Key Finding 1: the Technology department technical assessment
    # averages 11.3 days versus 4.2 days for all other departments.
    # This is the primary bottleneck in the hiring funnel.
    if dept == 'Technology':
        return max(4, min(22, int(np.random.normal(11.3, 3.2))))
    else:
        return max(1, min(9,  int(np.random.normal(4.2,  1.4))))


def get_stage_durations(dept: str, level: str, source: str) -> dict:
    # Key Finding 2: referral candidates move through the funnel
    # 38% faster than average (24 days vs 38 days time-to-hire).
    # The 0.62 multiplier is set to produce that ratio.
    ref_speed = 0.62 if source == 'Employee Referral' else 1.0

    return {
        'Resume Screen':         max(1, int(random.randint(2, 6)  * ref_speed)),
        'Recruiter Screen':      max(1, int(random.randint(3, 8)  * ref_speed)),
        'Hiring Manager Screen': max(1, int(random.randint(4, 10) * ref_speed)),
        'Technical Assessment':  max(1, int(get_tech_assess_duration(dept) * ref_speed)),
        'Technical Interview':   max(1, int(random.randint(3, 8)  * ref_speed)),
        'Final Panel Interview': max(1, int(random.randint(3, 8)  * ref_speed)),
        'Reference Check':       max(1, int(random.randint(3, 7)  * ref_speed)),
        'Offer':                 max(1, int(random.randint(1, 4)  * ref_speed)),
    }


print('Helper functions defined.')

Helper functions defined.


---
## Cell 5: GENERATE JOBS TABLE

In [6]:
# =============================================================
# CELL 5: GENERATE JOBS TABLE
# =============================================================
# 45 job requisitions distributed across departments.
# Status mix: ~65% filled, ~20% open, ~9% cancelled, ~6% on hold.
# =============================================================

def generate_jobs() -> pd.DataFrame:
    jobs = []

    status_pool = (['filled'] * 29 + ['open'] * 9 +
                   ['cancelled'] * 4 + ['on_hold'] * 3)
    random.shuffle(status_pool)

    job_counter = 0

    for dept, n_dept_jobs in DEPT_JOB_COUNTS.items():
        for i in range(n_dept_jobs):
            job_counter += 1
            job_id = f'JOB-{job_counter:03d}'
            level  = get_job_level_for_dept(dept)

            titles    = JOB_TITLES.get(dept, {}).get(level, [f'{dept} Specialist'])
            job_title = random.choice(titles)

            loc_roll = random.random()
            if loc_roll < 0.45:
                location = 'Vilnius, Lithuania'
            elif loc_roll < 0.65:
                location = 'Kaunas, Lithuania'
            elif loc_roll < 0.80:
                location = 'Remote - Europe'
            elif loc_roll < 0.88:
                location = 'Warsaw, Poland'
            elif loc_roll < 0.94:
                location = 'Berlin, Germany'
            else:
                location = 'London, United Kingdom'

            emp_type = 'contractor' if random.random() < 0.08 else 'full_time'

            months_ago = min(int(np.random.exponential(8)), 18)
            req_open   = TODAY - timedelta(days=months_ago * 30 + random.randint(-15, 15))
            req_open   = max(req_open, TALENTFLOW_GOLIVE)

            status = status_pool[job_counter - 1]

            if status == 'filled':
                days_to_fill = max(20, min(90, int(np.random.normal(42, 12))))
                req_close    = req_open + timedelta(days=days_to_fill)
                if req_close > TODAY:
                    req_close = TODAY - timedelta(days=random.randint(5, 30))
                target_fill  = req_open + timedelta(days=30)
            elif status == 'open':
                req_close   = None
                target_fill = TODAY + timedelta(days=random.randint(14, 45))
            elif status == 'cancelled':
                req_close = req_open + timedelta(days=random.randint(14, 60))
                if req_close > TODAY:
                    req_close = TODAY - timedelta(days=5)
                target_fill = req_open + timedelta(days=30)
            else:
                req_close   = None
                target_fill = TODAY + timedelta(days=60)

            hm_id  = f'HM-{random.randint(1, 25):03d}'
            rec_id = f'REC-{random.randint(1, 7):03d}'
            hc_approved = 2 if (level in ['ic1', 'ic2'] and random.random() < 0.15) else 1

            jobs.append({
                'job_id':                job_id,
                'job_title':             job_title,
                'department':            dept,
                'location':              location,
                'employment_type':       emp_type,
                'requisition_open_date': to_iso(req_open),
                'requisition_close_date': to_iso(req_close),
                'hiring_manager_id':     hm_id,
                'recruiter_id':          rec_id,
                'headcount_approved':    hc_approved,
                'status':                status,
                'target_fill_date':      to_iso(target_fill),
                'job_level':             level,
            })

    return pd.DataFrame(jobs)


jobs_df = generate_jobs()
print(f'Jobs generated: {len(jobs_df)} rows')
print(jobs_df['status'].value_counts().to_string())
print(jobs_df['department'].value_counts().to_string())

Jobs generated: 45 rows
status
filled       29
open          9
cancelled     4
on_hold       3
department
Technology              15
Commercial               7
Product                  6
Data and Analytics       5
Operations               4
Finance                  3
People                   3
Legal and Compliance     2


---
## Cell 6: GENERATE CANDIDATES TABLE

In [7]:
# =============================================================
# CELL 6: GENERATE CANDIDATES TABLE
# =============================================================
# 4,200 candidate records. Most applicants are from Lithuania
# and Poland reflecting the local and nearby talent markets.
# =============================================================

def generate_candidates(n: int) -> pd.DataFrame:
    candidates  = []
    used_emails = set()

    nationalities  = ['Lithuania', 'Poland', 'Ukraine', 'Germany',
                      'United Kingdom', 'Estonia', 'Latvia', 'Other']
    nat_weights    = [0.30, 0.20, 0.15, 0.10, 0.08, 0.06, 0.05, 0.06]

    cand_locations = [
        ('Vilnius', 'Lithuania'), ('Warsaw', 'Poland'), ('Kaunas', 'Lithuania'),
        ('Berlin', 'Germany'), ('Krakow', 'Poland'), ('Tallinn', 'Estonia'),
        ('London', 'United Kingdom'), ('Kyiv', 'Ukraine'), ('Riga', 'Latvia'),
        ('Amsterdam', 'Netherlands'), ('Hamburg', 'Germany'), ('Wroclaw', 'Poland'),
        ('Gdansk', 'Poland'), ('Lviv', 'Ukraine'), ('Kharkiv', 'Ukraine'),
    ]
    loc_weights = [0.22, 0.12, 0.10, 0.08, 0.07, 0.07, 0.07, 0.06, 0.05,
                   0.04, 0.04, 0.03, 0.02, 0.02, 0.01]

    for i in range(n):
        cand_id     = f'CAND-{i+1:04d}'
        nationality = random.choices(nationalities, weights=nat_weights, k=1)[0]
        first, last, _ = get_name(nationality)

        domain     = random.choice(['gmail.com', 'gmail.com', 'gmail.com',
                                    'outlook.com', 'yahoo.com', 'protonmail.com'])
        base_email = f'{first.lower()}.{last.lower()}@{domain}'
        email      = base_email
        attempt    = 1
        while email in used_emails:
            email   = f'{first.lower()}.{last.lower()}{attempt}@{domain}'
            attempt += 1
        used_emails.add(email)

        phone    = fake_generic.phone_number() if random.random() < 0.75 else None
        city, country = random.choices(cand_locations, weights=loc_weights, k=1)[0]

        company = random.choice(COMPANIES)
        current_title = None
        if company is not None:
            dept_  = random.choice(DEPARTMENTS)
            level_ = get_job_level_for_dept(dept_)
            current_title = random.choice(
                JOB_TITLES.get(dept_, {}).get(level_, ['Specialist']))

        linkedin = (f'linkedin.com/in/{first.lower()}-{last.lower()}-'
                    f'{random.randint(100, 999)}') if random.random() < 0.70 else None

        created = rand_date(TALENTFLOW_GOLIVE, TODAY)

        candidates.append({
            'candidate_id':    cand_id,
            'first_name':      first,
            'last_name':       last,
            'email':           email,
            'phone':           phone,
            'location_city':   city,
            'location_country': country,
            'current_company': company,
            'current_title':   current_title,
            'linkedin_url':    linkedin,
            'created_date':    to_iso(created),
        })

    return pd.DataFrame(candidates)


candidates_df = generate_candidates(N_APPLICATIONS)
print(f'Candidates generated: {len(candidates_df)} rows')
print(f'  With phone: {candidates_df["phone"].notna().sum()}')
print(f'  With LinkedIn: {candidates_df["linkedin_url"].notna().sum()}')

Candidates generated: 4200 rows
  With phone: 3130
  With LinkedIn: 2986


---
## Cell 7: GENERATE APPLICATIONS AND PIPELINE EVENTS

In [8]:
# =============================================================
# CELL 7: GENERATE APPLICATIONS AND PIPELINE EVENTS
# =============================================================
# Generates both tables together because pipeline events are
# tightly coupled to applications.
#
# Strategy: generate the 180 hired pipelines first with exact
# control over timings, then fill the remaining 4,020
# applications with rejections at each funnel stage.
# This guarantees the key findings are in the data rather
# than left to chance in a random simulation.
#
# Key findings embedded:
#   - 50 of 180 hires (28%) from Employee Referral
#   - Technology time-to-hire averages 47 days vs 38 overall
#   - Technical assessment is the primary bottleneck
# =============================================================

def generate_applications_and_pipeline(
    jobs_df: pd.DataFrame,
    candidates_df: pd.DataFrame
) -> tuple:

    applications    = []
    pipeline_events = []

    app_counter   = 0
    event_counter = 0

    # Hire targets by department. Technology gets the most because
    # it is the largest team and has the highest hiring velocity
    # post-Series B.
    dept_hire_targets = {
        'Technology':           70,
        'Product':              20,
        'Data and Analytics':   18,
        'Commercial':           25,
        'Finance':               8,
        'Operations':           15,
        'People':               12,
        'Legal and Compliance':  7,
    }

    hired_candidate_ids     = set()
    cand_idx                = 0
    referral_hire_count     = 0
    TARGET_REFERRAL_HIRES   = 50
    total_hires_generated   = 0

    # Pass 1: generate hired candidate pipelines
    for dept, target_hires in dept_hire_targets.items():
        dept_jobs = jobs_df[jobs_df['department'] == dept].to_dict('records')
        if not dept_jobs:
            continue

        hires_per_cycle = max(1, target_hires // len(dept_jobs))
        hires_in_dept   = 0

        for job in dept_jobs:
            if hires_in_dept >= target_hires:
                break

            n_hires = min(
                hires_per_cycle + random.randint(-1, 2),
                target_hires - hires_in_dept
            )
            n_hires = max(1, n_hires)

            job_req_open = date.fromisoformat(job['requisition_open_date'])
            level        = job['job_level']
            stages       = get_stage_sequence(dept, level)

            for h in range(n_hires):
                if cand_idx >= len(candidates_df):
                    break

                candidate = candidates_df.iloc[cand_idx].to_dict()
                cand_idx += 1

                app_counter += 1
                app_id       = f'APP-{app_counter:05d}'

                if referral_hire_count < TARGET_REFERRAL_HIRES:
                    source = 'Employee Referral'
                    referral_hire_count += 1
                else:
                    non_ref_sources = [s for s in SOURCES if s != 'Employee Referral']
                    non_ref_weights = [w for s, w in zip(SOURCES, SOURCE_WEIGHTS)
                                       if s != 'Employee Referral']
                    total_w         = sum(non_ref_weights)
                    non_ref_weights = [w / total_w for w in non_ref_weights]
                    source          = random.choices(non_ref_sources,
                                                     weights=non_ref_weights, k=1)[0]

                subtype      = SOURCE_SUBTYPES.get(source, 'Unknown')
                app_date     = days_later(job_req_open, 3, 45)
                if app_date > TODAY:
                    app_date = TODAY - timedelta(days=5)

                durations    = get_stage_durations(dept, level, source)
                current_date = app_date
                prev_stage   = 'Applied'

                for stage in stages:
                    days_in_prev = durations.get(prev_stage, random.randint(2, 6))
                    current_date = current_date + timedelta(days=days_in_prev)
                    if current_date > TODAY:
                        current_date = TODAY - timedelta(days=1)

                    event_counter += 1
                    pipeline_events.append({
                        'event_id':       f'EVT-{event_counter:05d}',
                        'application_id': app_id,
                        'from_stage':     prev_stage,
                        'to_stage':       stage,
                        'event_date':     to_iso(current_date),
                        'event_type':     'advanced',
                        'rejection_reason': None,
                        'interviewer_id': (f'INT-{random.randint(1, 40):03d}'
                                          if stage not in ['Applied', 'Resume Screen',
                                                           'Technical Assessment', 'Offer']
                                          else None),
                        'notes': None,
                    })
                    prev_stage = stage

                # Offer Accepted event
                offer_accept_date = current_date + timedelta(days=random.randint(1, 4))
                event_counter += 1
                pipeline_events.append({
                    'event_id':         f'EVT-{event_counter:05d}',
                    'application_id':   app_id,
                    'from_stage':       'Offer',
                    'to_stage':         'Offer Accepted',
                    'event_date':       to_iso(offer_accept_date),
                    'event_type':       'advanced',
                    'rejection_reason': None,
                    'interviewer_id':   None,
                    'notes':            None,
                })

                # Hired event
                start_date = offer_accept_date + timedelta(days=random.randint(14, 60))
                event_counter += 1
                pipeline_events.append({
                    'event_id':         f'EVT-{event_counter:05d}',
                    'application_id':   app_id,
                    'from_stage':       'Offer Accepted',
                    'to_stage':         'Hired',
                    'event_date':       to_iso(start_date),
                    'event_type':       'advanced',
                    'rejection_reason': None,
                    'interviewer_id':   None,
                    'notes':            None,
                })

                applications.append({
                    'application_id':    app_id,
                    'candidate_id':      candidate['candidate_id'],
                    'job_id':            job['job_id'],
                    'source':            source,
                    'source_subtype':    subtype,
                    'application_date':  to_iso(app_date),
                    'current_stage':     'Hired',
                    'current_stage_date': to_iso(start_date),
                    'hired':             True,
                    'rejected':          False,
                    'withdrawn':         False,
                })

                hired_candidate_ids.add(candidate['candidate_id'])
                hires_in_dept         += 1
                total_hires_generated += 1

    print(f'  Hired pipelines: {total_hires_generated}')
    print(f'  Referral hires: {referral_hire_count} '
          f'({referral_hire_count/total_hires_generated:.1%})')

    # Pass 2: generate rejected and withdrawn pipelines.
    # Distribution of rejection stages reflects a realistic funnel
    # where most candidates drop at the resume screen.
    rejection_stages = [
        ('Resume Screen',         'rejected',  0.45),
        ('Recruiter Screen',      'rejected',  0.20),
        ('Hiring Manager Screen', 'rejected',  0.12),
        ('Technical Assessment',  'withdrawn', 0.05),
        ('Technical Assessment',  'rejected',  0.05),
        ('Technical Interview',   'rejected',  0.04),
        ('Final Panel Interview', 'rejected',  0.05),
        ('Reference Check',       'rejected',  0.01),
        ('Offer',                 'withdrawn', 0.02),
        ('Recruiter Screen',      'withdrawn', 0.01),
    ]
    rej_stage_list = [s for s, _, _ in rejection_stages]
    rej_type_list  = [t for _, t, _ in rejection_stages]
    rej_weights    = [w for _, _, w in rejection_stages]

    remaining_cands = N_APPLICATIONS - total_hires_generated
    all_jobs        = jobs_df.to_dict('records')

    for i in range(remaining_cands):
        if cand_idx >= len(candidates_df):
            break

        candidate = candidates_df.iloc[cand_idx].to_dict()
        cand_idx += 1

        job   = random.choice(all_jobs)
        dept  = job['department']
        level = job['job_level']

        app_counter += 1
        app_id = f'APP-{app_counter:05d}'

        source  = random.choices(SOURCES, weights=SOURCE_WEIGHTS, k=1)[0]
        subtype = SOURCE_SUBTYPES.get(source, 'Unknown')

        job_req_open = date.fromisoformat(job['requisition_open_date'])
        app_date     = days_later(job_req_open, 1, 60)
        if app_date > TODAY:
            app_date = TODAY - timedelta(days=random.randint(1, 30))

        rej_idx   = random.choices(range(len(rejection_stages)),
                                   weights=rej_weights, k=1)[0]
        rej_stage = rej_stage_list[rej_idx]
        rej_type  = rej_type_list[rej_idx]

        rej_stage_order = PIPELINE_STAGES_ORDERED.index(rej_stage)

        current_date = app_date
        prev_stage   = 'Applied'
        last_stage   = 'Applied'
        last_date    = app_date
        durations    = get_stage_durations(dept, level, source)

        for stage in get_stage_sequence(dept, level):
            stage_order = PIPELINE_STAGES_ORDERED.index(stage)
            if stage_order > rej_stage_order:
                break

            days_in_prev = durations.get(prev_stage, random.randint(2, 5))
            current_date = current_date + timedelta(days=days_in_prev)
            if current_date > TODAY:
                current_date = TODAY - timedelta(days=random.randint(0, 3))

            event_counter += 1
            is_rejection   = (stage == rej_stage)

            if is_rejection:
                reason = None
                if rej_type == 'rejected':
                    reasons = REJECTION_REASONS.get(stage,
                                                    ['Preferred another candidate'])
                    reason  = random.choice(reasons)
                elif rej_type == 'withdrawn':
                    reason = random.choice(WITHDRAWAL_REASONS)

                # DQ-ATS-08: 35% of rejections have no reason logged.
                # This is injected here rather than in Cell 13 because
                # it reflects a process gap, not a data entry error.
                if random.random() < 0.35:
                    reason = None

                pipeline_events.append({
                    'event_id':         f'EVT-{event_counter:05d}',
                    'application_id':   app_id,
                    'from_stage':       prev_stage,
                    'to_stage':         stage,
                    'event_date':       to_iso(current_date),
                    'event_type':       rej_type,
                    'rejection_reason': reason,
                    'interviewer_id':   None,
                    'notes':            None,
                })
                last_stage = stage
                last_date  = current_date
                break
            else:
                pipeline_events.append({
                    'event_id':         f'EVT-{event_counter:05d}',
                    'application_id':   app_id,
                    'from_stage':       prev_stage,
                    'to_stage':         stage,
                    'event_date':       to_iso(current_date),
                    'event_type':       'advanced',
                    'rejection_reason': None,
                    'interviewer_id':   (f'INT-{random.randint(1, 40):03d}'
                                        if stage not in ['Applied', 'Resume Screen',
                                                         'Technical Assessment', 'Offer']
                                        else None),
                    'notes': None,
                })
                last_stage = stage
                last_date  = current_date
                prev_stage = stage

        applications.append({
            'application_id':    app_id,
            'candidate_id':      candidate['candidate_id'],
            'job_id':            job['job_id'],
            'source':            source,
            'source_subtype':    subtype,
            'application_date':  to_iso(app_date),
            'current_stage':     last_stage,
            'current_stage_date': to_iso(last_date),
            'hired':             False,
            'rejected':          (rej_type == 'rejected'),
            'withdrawn':         (rej_type == 'withdrawn'),
        })

    return pd.DataFrame(applications), pd.DataFrame(pipeline_events), hired_candidate_ids


print('Generating applications and pipeline events...')
applications_df, pipeline_events_df, hired_candidate_ids = \
    generate_applications_and_pipeline(jobs_df, candidates_df)

print(f'\nApplications: {len(applications_df)} rows')
print(f'Pipeline events: {len(pipeline_events_df)} rows')
print(f'Hires: {applications_df["hired"].sum()}')

src_counts     = applications_df['source'].value_counts()
hired_by_src   = applications_df[applications_df['hired']]['source'].value_counts()
print('\nSource breakdown:')
for src in SOURCES:
    n_apps  = src_counts.get(src, 0)
    n_hires = hired_by_src.get(src, 0)
    if n_apps > 0:
        print(f'  {src:<25} {n_apps:>4} apps  {n_hires:>3} hires '
              f'({n_hires/len(applications_df[applications_df["hired"]]):.1%} of hires)')

Generating applications and pipeline events...
  Hired pipelines: 166
  Referral hires: 50 (30.1%)

Applications: 4200 rows
Pipeline events: 14399 rows
Hires: 166

Source breakdown:
  LinkedIn Organic          1220 apps   46 hires (27.7% of hires)
  LinkedIn Paid              263 apps    8 hires (4.8% of hires)
  LinkedIn Recruiter         756 apps   19 hires (11.4% of hires)
  Employee Referral          484 apps   50 hires (30.1% of hires)
  Direct Application         476 apps   18 hires (10.8% of hires)
  Indeed                     333 apps    6 hires (3.6% of hires)
  Glassdoor                  244 apps    9 hires (5.4% of hires)
  Recruitment Agency         183 apps    6 hires (3.6% of hires)
  GitHub Sourcing             75 apps    1 hires (0.6% of hires)
  University Partnership      74 apps    2 hires (1.2% of hires)
  Other                       92 apps    1 hires (0.6% of hires)


---
## Cell 8: GENERATE OFFERS TABLE

In [9]:
# =============================================================
# CELL 8: GENERATE OFFERS TABLE
# =============================================================
# 200 offers total: 180 accepted (the hires) and 20 declined,
# rescinded, or expired. Acceptance rate = 90%.
# =============================================================

def generate_offers(
    applications_df: pd.DataFrame,
    jobs_df: pd.DataFrame,
    pipeline_events_df: pd.DataFrame
) -> pd.DataFrame:

    offers = []

    hired_apps = applications_df[applications_df['hired']].copy()
    offer_events = pipeline_events_df[
        pipeline_events_df['to_stage'] == 'Offer'
    ][['application_id', 'event_date']].rename(
        columns={'event_date': 'offer_event_date'}
    )
    hired_apps   = hired_apps.merge(offer_events, on='application_id', how='left')
    jobs_lookup  = jobs_df.set_index('job_id')[['job_level', 'location']].to_dict('index')

    offer_counter = 0

    for _, app in hired_apps.iterrows():
        offer_counter += 1
        offer_id = f'OFR-{offer_counter:03d}'

        job_info = jobs_lookup.get(app['job_id'], {})
        level    = job_info.get('job_level', 'ic2')
        location = job_info.get('location', 'Vilnius, Lithuania')

        if 'Lithuania' in location:   country = 'Lithuania'
        elif 'Poland'   in location:  country = 'Poland'
        elif 'Germany'  in location:  country = 'Germany'
        elif 'United Kingdom' in location: country = 'United Kingdom'
        else:                         country = 'Lithuania'

        salary, currency = get_salary(country, level)

        offer_date_str = app.get('offer_event_date')
        if pd.notna(offer_date_str) and offer_date_str:
            offer_date = date.fromisoformat(str(offer_date_str)[:10])
        else:
            offer_date = date.fromisoformat(app['current_stage_date'][:10])

        accepted_date = offer_date + timedelta(days=random.randint(1, 4))

        offers.append({
            'offer_id':       offer_id,
            'application_id': app['application_id'],
            'offer_date':     to_iso(offer_date),
            'offer_amount':   salary,
            'offer_currency': currency,
            'equity_offered': random.random() < 0.35,
            'offer_status':   'accepted',
            'decline_reason': None,
            'accepted_date':  to_iso(accepted_date),
        })

    # 20 declined, rescinded, or expired offers
    declined_statuses = ['declined', 'declined', 'declined', 'rescinded', 'expired']

    for _ in range(20):
        offer_counter += 1
        offer_id = f'OFR-{offer_counter:03d}'
        base_app = hired_apps.sample(1).iloc[0]
        job_info = jobs_lookup.get(base_app['job_id'], {})
        level    = job_info.get('job_level', 'ic2')
        salary, currency = get_salary('Lithuania', level)
        status   = random.choice(declined_statuses)

        offers.append({
            'offer_id':       offer_id,
            'application_id': base_app['application_id'],
            'offer_date':     to_iso(TODAY - timedelta(days=random.randint(10, 180))),
            'offer_amount':   salary,
            'offer_currency': currency,
            'equity_offered': random.random() < 0.35,
            'offer_status':   status,
            'decline_reason': (random.choice(OFFER_DECLINE_REASONS)
                               if status in ['declined', 'rescinded'] else None),
            'accepted_date':  None,
        })

    return pd.DataFrame(offers)


offers_df = generate_offers(applications_df, jobs_df, pipeline_events_df)
print(f'Offers generated: {len(offers_df)} rows')
print(offers_df['offer_status'].value_counts().to_string())
print(f'Acceptance rate: {(offers_df["offer_status"]=="accepted").mean():.1%}')

Offers generated: 186 rows
offer_status
accepted     166
declined      13
expired        4
rescinded      3
Acceptance rate: 89.2%


---
## Cell 9: GENERATE EMPLOYEES TABLE

In [10]:
# =============================================================
# CELL 9: GENERATE EMPLOYEES TABLE
# =============================================================
# PeopleCore holds 300 active + 45 terminated employees.
#
# Two groups:
#   Group A: employees hired through TalentFlow (have ATS records)
#   Group B: pre-ATS employees who joined before Jan 2024
#
# The department names used here differ from TalentFlow names.
# That is data quality issue DQ-HRIS-02. The canonical mapping
# is in the cleaning pipeline.
# =============================================================

# PeopleCore department names differ from TalentFlow names.
# This is intentional to demonstrate cross-system data issues.
DEPT_MAP_HRIS = {
    'Technology':           'Engineering',
    'Product':              'Product',
    'Data and Analytics':   'Analytics',
    'Commercial':           'Growth',
    'Finance':              'Finance',
    'Operations':           'Operations',
    'People':               'HR',
    'Legal and Compliance': 'Legal',
}

SUB_DEPTS = {
    'Technology':           ['Backend', 'Frontend', 'Mobile', 'DevOps', 'QA', 'Security'],
    'Data and Analytics':   ['Data Engineering', 'Analytics', 'Data Science', 'BI'],
    'Commercial':           ['Performance Marketing', 'CRM', 'SEO', 'Brand', 'Growth'],
    'Operations':           ['Customer Support', 'Trust and Safety', 'Partner Ops'],
    'People':               ['Talent Acquisition', 'HR', 'L&D', 'Compensation'],
    'Product':              ['Core Product', 'Platform', 'UX Research'],
    'Finance':              ['FP&A', 'Accounting'],
    'Legal and Compliance': ['Legal', 'Compliance'],
}


def generate_employees(
    jobs_df, applications_df, candidates_df, hired_candidate_ids
) -> pd.DataFrame:

    employees   = []
    emp_counter = 0
    manager_ids = []

    n_total    = N_ACTIVE_EMPLOYEES + N_TERMINATED
    n_ats_hires = len(hired_candidate_ids)
    n_pre_ats   = n_total - n_ats_hires

    # Group A: ATS-sourced employees
    hired_apps  = applications_df[applications_df['hired']].copy()
    hired_cands = candidates_df[
        candidates_df['candidate_id'].isin(hired_candidate_ids)
    ].set_index('candidate_id')

    hire_events = pipeline_events_df[
        pipeline_events_df['to_stage'] == 'Hired'
    ][['application_id', 'event_date']].rename(
        columns={'event_date': 'start_date_str'})
    offer_events = pipeline_events_df[
        pipeline_events_df['to_stage'] == 'Offer Accepted'
    ][['application_id', 'event_date']].rename(
        columns={'event_date': 'hire_date_str'})

    hired_apps = hired_apps.merge(hire_events,  on='application_id', how='left')
    hired_apps = hired_apps.merge(offer_events, on='application_id', how='left')
    hired_apps = hired_apps.merge(
        jobs_df[['job_id', 'department', 'job_level', 'location']],
        on='job_id', how='left')

    for _, app in hired_apps.iterrows():
        emp_counter += 1
        emp_id = f'EMP-{emp_counter:04d}'

        cand_id  = app['candidate_id']
        cand     = hired_cands.loc[cand_id] if cand_id in hired_cands.index else None
        dept_ats = app['department']
        dept_hris = DEPT_MAP_HRIS.get(dept_ats, dept_ats)
        sub_dept  = random.choice(SUB_DEPTS.get(dept_ats, [None]))
        level     = app['job_level']

        job_loc = app.get('location', 'Vilnius, Lithuania')
        if 'Vilnius' in str(job_loc):          city, country = 'Vilnius', 'Lithuania'
        elif 'Kaunas' in str(job_loc):         city, country = 'Kaunas',  'Lithuania'
        elif 'Remote' in str(job_loc):         city, country = 'Remote',  'Lithuania'
        elif 'Warsaw' in str(job_loc):         city, country = 'Warsaw',  'Poland'
        elif 'Berlin' in str(job_loc):         city, country = 'Berlin',  'Germany'
        elif 'London' in str(job_loc):         city, country = 'London',  'United Kingdom'
        else:                                  city, country = 'Vilnius', 'Lithuania'

        salary, currency = get_salary(country, level)

        if cand is not None:
            first, last  = cand['first_name'], cand['last_name']
            nationality  = cand['location_country']
        else:
            nationality  = country
            first, last, _ = get_name(nationality)

        _, _, gender = get_name(nationality)

        # DQ-HRIS-01: 15% of CEE employees have monthly salary entered as annual.
        monthly_risk = SALARY_RANGES.get((country, level), (0, 0, 'EUR', False))[3]
        if monthly_risk and random.random() < 0.15:
            salary_amount = round(salary / 12 / 100) * 100
            salary_type   = 'annual'   # Wrong label - this is a monthly figure
        else:
            salary_amount = salary
            salary_type   = 'annual'

        hire_date_str  = app.get('hire_date_str')
        start_date_str = app.get('start_date_str')

        hire_date = (date.fromisoformat(str(hire_date_str)[:10])
                     if pd.notna(hire_date_str) and hire_date_str
                     else TALENTFLOW_GOLIVE + timedelta(days=random.randint(30, 400)))
        start_date = (date.fromisoformat(str(start_date_str)[:10])
                      if pd.notna(start_date_str) and start_date_str
                      else hire_date + timedelta(days=random.randint(14, 30)))

        is_terminated = (random.random() < 0.13
                         and start_date < TODAY - timedelta(days=180))

        if is_terminated:
            term_date = start_date + timedelta(days=random.randint(90, 500))
            if term_date > TODAY:
                term_date = TODAY - timedelta(days=random.randint(5, 60))
            term_reason = (None if random.random() < 0.30
                           else random.choice(['voluntary_resignation',
                                               'voluntary_resignation',
                                               'involuntary_performance',
                                               'involuntary_restructuring',
                                               'end_of_contract']))
            status = 'terminated'
        else:
            term_date, term_reason, status = None, None, 'active'

        mgr_id     = random.choice(manager_ids) if manager_ids else None
        work_email = f'{first.lower()}.{last.lower()}@enovatech.com'

        emp_row = {
            'employee_id':        emp_id,
            'first_name':         first,
            'last_name':          last,
            'email':              work_email,
            'department':         dept_hris,
            'sub_department':     sub_dept,
            'job_title':          random.choice(
                                      JOB_TITLES.get(dept_ats, {}).get(level, ['Specialist'])),
            'job_level':          level,
            'employment_type':    'full_time',
            'hire_date':          to_iso(hire_date),
            'start_date':         to_iso(start_date),
            'manager_id':         mgr_id,
            'location_city':      city,
            'location_country':   country,
            'salary_amount':      salary_amount,
            'salary_currency':    currency,
            'salary_type':        salary_type,
            'gender':             gender,
            'nationality':        nationality,
            'status':             status,
            'termination_date':   to_iso(term_date),
            'termination_reason': term_reason,
            'ats_candidate_id':   cand_id,
        }
        employees.append(emp_row)

        if level in ['manager', 'senior_manager', 'director', 'vp']:
            manager_ids.append(emp_id)

    # Group B: pre-ATS employees.
    # These joined before TalentFlow existed. They appear in PeopleCore
    # with no matching ATS record. The cleaning pipeline tags them
    # as pre_ats = True and excludes them from ATS-dependent metrics.
    pre_ats_dept_weights = {
        'Technology': 0.30, 'Product': 0.12, 'Commercial': 0.20,
        'Operations': 0.15, 'Finance': 0.08, 'People': 0.07,
        'Data and Analytics': 0.05, 'Legal and Compliance': 0.03,
    }
    pre_ats_depts   = list(pre_ats_dept_weights.keys())
    pre_ats_wts     = list(pre_ats_dept_weights.values())
    n_remaining     = n_total - len(employees)

    for j in range(n_remaining):
        emp_counter += 1
        emp_id   = f'EMP-{emp_counter:04d}'
        dept_ats = random.choices(pre_ats_depts, weights=pre_ats_wts, k=1)[0]
        dept_hris = DEPT_MAP_HRIS.get(dept_ats, dept_ats)
        sub_dept  = random.choice(SUB_DEPTS.get(dept_ats, [None]))
        level     = get_job_level_for_dept(dept_ats)

        country = random.choices(
            ['Lithuania', 'Poland', 'Ukraine', 'Germany', 'United Kingdom'],
            weights=[0.55, 0.18, 0.12, 0.08, 0.07], k=1)[0]
        city    = {'Lithuania': 'Vilnius', 'Poland': 'Warsaw',
                   'Ukraine': 'Kyiv', 'Germany': 'Berlin',
                   'United Kingdom': 'London'}.get(country, 'Vilnius')

        first, last, gender = get_name(country)
        salary, currency    = get_salary(country, level)

        monthly_risk = SALARY_RANGES.get((country, level), (0, 0, 'EUR', False))[3]
        if monthly_risk and random.random() < 0.15:
            salary_amount = round(salary / 12 / 100) * 100
            salary_type   = 'annual'
        else:
            salary_amount = salary
            salary_type   = 'annual'

        hire_date  = rand_date(COMPANY_FOUNDED, TALENTFLOW_GOLIVE - timedelta(days=1))
        start_date = hire_date + timedelta(days=random.randint(7, 30))

        is_terminated = random.random() < 0.20
        if is_terminated:
            term_date = start_date + timedelta(days=random.randint(60, 600))
            if term_date > TODAY:
                term_date = TODAY - timedelta(days=random.randint(10, 90))
            term_reason = (None if random.random() < 0.40
                           else random.choice(['voluntary_resignation',
                                               'voluntary_resignation',
                                               'involuntary_performance',
                                               'end_of_contract']))
            status = 'terminated'
        else:
            term_date, term_reason, status = None, None, 'active'

        mgr_id     = random.choice(manager_ids) if manager_ids else None
        work_email = f'{first.lower()}.{last.lower()}@enovatech.com'

        employees.append({
            'employee_id':        emp_id,
            'first_name':         first,
            'last_name':          last,
            'email':              work_email,
            'department':         dept_hris,
            'sub_department':     sub_dept,
            'job_title':          random.choice(
                                      JOB_TITLES.get(dept_ats, {}).get(level, ['Specialist'])),
            'job_level':          level,
            'employment_type':    random.choices(['full_time', 'contractor'],
                                                 weights=[0.92, 0.08], k=1)[0],
            'hire_date':          to_iso(hire_date),
            'start_date':         to_iso(start_date),
            'manager_id':         mgr_id,
            'location_city':      city,
            'location_country':   country,
            'salary_amount':      salary_amount,
            'salary_currency':    currency,
            'salary_type':        salary_type,
            'gender':             gender,
            'nationality':        country,
            'status':             status,
            'termination_date':   to_iso(term_date),
            'termination_reason': term_reason,
            'ats_candidate_id':   None,
        })

        if level in ['manager', 'senior_manager', 'director', 'vp']:
            manager_ids.append(emp_id)

    emps_df = pd.DataFrame(employees)

    if manager_ids:
        null_mask = emps_df['manager_id'].isna()
        emps_df.loc[null_mask, 'manager_id'] = [
            random.choice(manager_ids) for _ in range(null_mask.sum())
        ]

    return emps_df


employees_df = generate_employees(
    jobs_df, applications_df, candidates_df, hired_candidate_ids)

print(f'Employees generated: {len(employees_df)} rows')
print(f'  Active: {(employees_df["status"]=="active").sum()}')
print(f'  Terminated: {(employees_df["status"]=="terminated").sum()}')
print(f'  With ATS ID: {employees_df["ats_candidate_id"].notna().sum()} '
      f'({employees_df["ats_candidate_id"].notna().mean():.1%})')
print('\nDepartments (HRIS names):')
print(employees_df[employees_df['status']=='active']['department'].value_counts().to_string())

Employees generated: 345 rows
  Active: 301
  Terminated: 44
  With ATS ID: 166 (48.1%)

Departments (HRIS names):
department
Engineering    104
Growth          51
Operations      36
Product         33
Analytics       26
HR              19
Finance         18
Legal           14


---
## Cell 10: GENERATE PERFORMANCE REVIEWS

In [11]:
# =============================================================
# CELL 10: GENERATE PERFORMANCE REVIEWS
# =============================================================
# 280 reviews for employees with 6+ months tenure.
# Referral hires score 4.1 average vs 3.6 overall,
# which is Key Finding 2's quality-of-hire component.
# =============================================================

def generate_performance_reviews(
    employees_df: pd.DataFrame,
    applications_df: pd.DataFrame
) -> pd.DataFrame:

    reviews = []

    six_months_ago = TODAY - timedelta(days=180)
    eligible = employees_df[
        pd.to_datetime(employees_df['start_date']) <= pd.Timestamp(six_months_ago)
    ].copy()

    referral_cand_ids = set(
        applications_df[
            (applications_df['hired']) &
            (applications_df['source'] == 'Employee Referral')
        ]['candidate_id'].tolist()
    )
    referral_emp_ids = set(
        employees_df[
            employees_df['ats_candidate_id'].isin(referral_cand_ids)
        ]['employee_id'].tolist()
    )

    review_periods = ['H1_2024', 'H2_2024', 'H1_2025']
    reviewer_pool  = employees_df[
        employees_df['job_level'].isin(['manager', 'senior_manager', 'director', 'vp'])
    ]['employee_id'].tolist()
    if not reviewer_pool:
        reviewer_pool = employees_df['employee_id'].tolist()[:20]

    rev_counter = 0

    for _, emp in eligible.sample(
            min(N_PERF_REVIEWS, len(eligible)), random_state=RANDOM_SEED).iterrows():

        rev_counter += 1

        # Referral employees score higher on average (Key Finding 2).
        # Mean 4.1 for referrals vs 3.6 for non-referrals.
        if emp['employee_id'] in referral_emp_ids:
            rating = random.choices([2, 3, 4, 4, 5, 5],
                                    weights=[0.02, 0.10, 0.35, 0.30, 0.15, 0.08], k=1)[0]
        else:
            rating = random.choices([1, 2, 3, 4, 5],
                                    weights=[0.03, 0.10, 0.35, 0.38, 0.14], k=1)[0]

        period = random.choice(review_periods)
        year   = int(period.split('_')[1])
        month  = random.randint(6, 7) if 'H1' in period else random.randint(11, 12)
        submitted = date(year, month, random.randint(1, 28))
        if submitted > TODAY:
            submitted = TODAY - timedelta(days=random.randint(5, 30))

        reviews.append({
            'review_id':      f'REV-{rev_counter:04d}',
            'employee_id':    emp['employee_id'],
            'review_period':  period,
            'rating':         rating,
            'reviewer_id':    random.choice(reviewer_pool),
            'submitted_date': to_iso(submitted),
        })

    return pd.DataFrame(reviews)


performance_df = generate_performance_reviews(employees_df, applications_df)
print(f'Performance reviews: {len(performance_df)} rows')
print(f'Average rating: {performance_df["rating"].mean():.2f}')
print(performance_df['rating'].value_counts().sort_index().to_string())

Performance reviews: 222 rows
Average rating: 3.53
rating
1     5
2    18
3    80
4    92
5    27


---
## Cell 11: GENERATE COMPENSATION CHANGES

In [12]:
# =============================================================
# CELL 11: GENERATE COMPENSATION CHANGES
# =============================================================
# 120 salary change events for employees with 12+ months tenure.
# Used in the SQL analysis layer to track pay progression.
# =============================================================

def generate_compensation_changes(employees_df: pd.DataFrame) -> pd.DataFrame:
    changes = []

    twelve_months_ago = TODAY - timedelta(days=365)
    eligible = employees_df[
        pd.to_datetime(employees_df['start_date']) <= pd.Timestamp(twelve_months_ago)
    ].copy()

    sampled = eligible.sample(
        min(N_COMP_CHANGES, len(eligible)), random_state=RANDOM_SEED + 1)

    change_reasons = ['merit_increase', 'merit_increase', 'merit_increase',
                      'promotion', 'promotion', 'market_adjustment', 'role_change']

    chg_counter = 0

    for _, emp in sampled.iterrows():
        chg_counter += 1
        old_salary = emp['salary_amount']
        reason     = random.choice(change_reasons)

        if reason == 'promotion':          pct = random.uniform(0.15, 0.30)
        elif reason == 'merit_increase':   pct = random.uniform(0.04, 0.12)
        elif reason == 'market_adjustment': pct = random.uniform(0.05, 0.18)
        else:                              pct = random.uniform(0.08, 0.20)

        new_salary = round(old_salary * (1 + pct) / 500) * 500

        start = date.fromisoformat(emp['start_date'][:10])
        change_date = start + timedelta(
            days=random.randint(180, min(730, (TODAY - start).days)))

        changes.append({
            'change_id':     f'CHG-{chg_counter:04d}',
            'employee_id':   emp['employee_id'],
            'change_date':   to_iso(change_date),
            'old_salary':    old_salary,
            'new_salary':    new_salary,
            'change_reason': reason,
        })

    return pd.DataFrame(changes)


compensation_df = generate_compensation_changes(employees_df)
print(f'Compensation changes: {len(compensation_df)} rows')
print(compensation_df['change_reason'].value_counts().to_string())

Compensation changes: 120 rows
change_reason
merit_increase       50
promotion            32
role_change          22
market_adjustment    16


---
## Cell 12: GENERATE HEADCOUNT PLAN

In [13]:
# =============================================================
# CELL 12: GENERATE HEADCOUNT PLAN
# =============================================================
# Finance uses its own department taxonomy that combines some
# departments and renames others. That is data quality issue
# DQ-PLAN-01. Two additional issues are injected:
#   DQ-PLAN-02: missing Q4 data for two departments
#   DQ-PLAN-03: Engineering Q2 plan vs actual discrepancy
#               due to emergency headcount approvals
# =============================================================

def generate_headcount_plan(employees_df: pd.DataFrame) -> pd.DataFrame:

    # Finance combines Technology + Product into "Engineering & Product"
    # and People + Legal into "HR & Legal". Neither mapping is obvious
    # from the names alone, which makes cross-system analysis harder.
    HRIS_TO_FINANCE = {
        'Engineering': 'Engineering & Product',
        'Product':     'Engineering & Product',
        'Analytics':   'Analytics',
        'Growth':      'Sales & Marketing',
        'Finance':     'Finance',
        'Operations':  'Operations',
        'HR':          'HR & Legal',
        'Legal':       'HR & Legal',
    }

    quarters = ['Q3_2023', 'Q4_2023', 'Q1_2024', 'Q2_2024',
                'Q3_2024', 'Q4_2024', 'Q1_2025', 'Q2_2025']

    planned_targets = {
        ('Engineering & Product', 'Q3_2023'): 28,
        ('Engineering & Product', 'Q4_2023'): 35,
        ('Engineering & Product', 'Q1_2024'): 50,
        ('Engineering & Product', 'Q2_2024'): 85,  # DQ-PLAN-03: actual was 92
        ('Engineering & Product', 'Q3_2024'): 110,
        ('Engineering & Product', 'Q4_2024'): 140,
        ('Engineering & Product', 'Q1_2025'): 155,
        ('Engineering & Product', 'Q2_2025'): 165,
        ('Analytics',    'Q3_2023'):  5, ('Analytics',    'Q4_2023'):  8,
        ('Analytics',    'Q1_2024'): 12, ('Analytics',    'Q2_2024'): 18,
        ('Analytics',    'Q3_2024'): 25, ('Analytics',    'Q4_2024'): 32,
        ('Analytics',    'Q1_2025'): 36, ('Analytics',    'Q2_2025'): 40,
        ('Sales & Marketing', 'Q3_2023'):  8, ('Sales & Marketing', 'Q4_2023'): 12,
        ('Sales & Marketing', 'Q1_2024'): 18, ('Sales & Marketing', 'Q2_2024'): 28,
        ('Sales & Marketing', 'Q3_2024'): 38, ('Sales & Marketing', 'Q4_2024'): 48,
        ('Sales & Marketing', 'Q1_2025'): 52, ('Sales & Marketing', 'Q2_2025'): 56,
        ('Finance',      'Q3_2023'):  4, ('Finance',      'Q4_2023'):  5,
        ('Finance',      'Q1_2024'):  7, ('Finance',      'Q2_2024'): 10,
        ('Finance',      'Q3_2024'): 13, ('Finance',      'Q4_2024'): 16,
        ('Finance',      'Q1_2025'): 17, ('Finance',      'Q2_2025'): 18,
        ('Operations',   'Q3_2023'):  6, ('Operations',   'Q4_2023'):  9,
        ('Operations',   'Q1_2024'): 14, ('Operations',   'Q2_2024'): 22,
        ('Operations',   'Q3_2024'): 30,
        ('Operations',   'Q4_2024'): None,  # DQ-PLAN-02: not submitted
        ('Operations',   'Q1_2025'): 38, ('Operations',   'Q2_2025'): 42,
        ('HR & Legal',   'Q3_2023'):  4, ('HR & Legal',   'Q4_2023'):  6,
        ('HR & Legal',   'Q1_2024'):  9, ('HR & Legal',   'Q2_2024'): 14,
        ('HR & Legal',   'Q3_2024'): 18,
        ('HR & Legal',   'Q4_2024'): None,  # DQ-PLAN-02: not submitted
        ('HR & Legal',   'Q1_2025'): 20, ('HR & Legal',   'Q2_2025'): 22,
    }

    quarter_end_dates = {
        'Q3_2023': date(2023, 9, 30), 'Q4_2023': date(2023, 12, 31),
        'Q1_2024': date(2024, 3, 31), 'Q2_2024': date(2024, 6, 30),
        'Q3_2024': date(2024, 9, 30), 'Q4_2024': date(2024, 12, 31),
        'Q1_2025': date(2025, 3, 31), 'Q2_2025': date(2025, 6, 30),
    }

    plan_rows   = []
    plan_counter = 0

    for (fin_dept, quarter), approved in planned_targets.items():
        plan_counter += 1
        q_end = quarter_end_dates[quarter]

        if q_end <= TODAY:
            active_at_q_end = employees_df[
                (pd.to_datetime(employees_df['start_date']) <= pd.Timestamp(q_end)) &
                (
                    employees_df['termination_date'].isna() |
                    (pd.to_datetime(employees_df['termination_date']) > pd.Timestamp(q_end))
                ) &
                (employees_df['department'].map(
                    lambda d: HRIS_TO_FINANCE.get(d, 'Other')) == fin_dept)
            ]
            actual = len(active_at_q_end)
        else:
            actual = None

        note = None
        if fin_dept == 'Engineering & Product' and quarter == 'Q2_2024':
            note = ('Actual includes 7 emergency headcount approvals not in standard plan. '
                    'Plan = 85, emergency = 7, total approved = 92, actual = 92, variance = 0.')
        elif approved is None:
            note = 'Q4 plan not submitted by department head at time of Finance export.'

        plan_rows.append({
            'plan_id':            f'PLAN-{plan_counter:03d}',
            'department':          fin_dept,
            'quarter':             quarter,
            'headcount_approved':  approved,
            'headcount_actual':    actual,
            'notes':               note,
        })

    return pd.DataFrame(plan_rows)


headcount_plan_df = generate_headcount_plan(employees_df)
print(f'Headcount plan: {len(headcount_plan_df)} rows')
print(headcount_plan_df[['department', 'quarter',
                          'headcount_approved', 'headcount_actual']].to_string())

Headcount plan: 48 rows
               department  quarter  headcount_approved  headcount_actual
0   Engineering & Product  Q3_2023                28.0                42
1   Engineering & Product  Q4_2023                35.0                64
2   Engineering & Product  Q1_2024                50.0                69
3   Engineering & Product  Q2_2024                85.0                79
4   Engineering & Product  Q3_2024               110.0                79
5   Engineering & Product  Q4_2024               140.0                83
6   Engineering & Product  Q1_2025               155.0                88
7   Engineering & Product  Q2_2025               165.0                90
8               Analytics  Q3_2023                 5.0                 9
9               Analytics  Q4_2023                 8.0                13
10              Analytics  Q1_2024                12.0                12
11              Analytics  Q2_2024                18.0                15
12              Analytics  

---
## Cell 13: INJECT DATA QUALITY ISSUES

In [14]:
# =============================================================
# CELL 13: INJECT DATA QUALITY ISSUES
# =============================================================
# Introduces deliberate messiness into the clean data to
# simulate real-world ATS and HRIS export quality.
#
# Every issue injected here is documented in
# docs/step1_foundations.md with its cleaning rule.
# The cleaning pipeline in Step 3 is written specifically
# to find and fix each of these.
# =============================================================

def inject_dq_issues(jobs_df, candidates_df, applications_df,
                     pipeline_events_df, offers_df):

    jobs   = jobs_df.copy()
    cands  = candidates_df.copy()
    apps   = applications_df.copy()
    events = pipeline_events_df.copy()
    offers = offers_df.copy()

    # DQ-ATS-01: Inconsistent stage naming (~15% of pipeline events)
    stage_variants = {
        'Recruiter Screen':      ['Phone Screen', 'phone_screen', 'Phone Interview',
                                  'PHONE SCREEN', 'Recruiter Call', 'recruiter_screen'],
        'Hiring Manager Screen': ['HM Screen', 'HM Call', 'hiring_manager_screen',
                                  'Manager Screen'],
        'Technical Assessment':  ['Tech Assessment', 'technical_assessment',
                                  'Take Home Task', 'Coding Challenge'],
        'Technical Interview':   ['Tech Interview', 'technical_interview', 'Live Coding'],
        'Final Panel Interview': ['Panel', 'Final Panel', 'Final Interview',
                                  'final_panel_interview'],
        'Reference Check':       ['Refs', 'reference_check', 'References'],
        'Resume Screen':         ['CV Review', 'Resume Review', 'CV Screen', 'resume_screen'],
    }

    dq01_mask = events.sample(frac=0.15, random_state=RANDOM_SEED).index
    for idx in dq01_mask:
        for col in ['from_stage', 'to_stage']:
            original = events.at[idx, col]
            if original in stage_variants:
                events.at[idx, col] = random.choice(stage_variants[original])
    print(f'DQ-ATS-01: {len(dq01_mask)} pipeline event rows affected')

    # DQ-ATS-02: Mixed date formats (~8% of date fields)
    def corrupt_date(val, style):
        if pd.isna(val) or val is None:
            return val
        try:
            from datetime import date as dt
            d = dt.fromisoformat(str(val)[:10])
            return d.strftime('%d/%m/%Y') if style == 'dmy' else d.strftime('%m/%d/%Y')
        except:
            return val

    for col in ['application_date', 'current_stage_date']:
        mask = apps.sample(frac=0.04, random_state=RANDOM_SEED + 1).index
        for idx in mask:
            apps.at[idx, col] = corrupt_date(
                apps.at[idx, col], random.choice(['dmy', 'mdy']))

    mask = events.sample(frac=0.04, random_state=RANDOM_SEED + 2).index
    for idx in mask:
        events.at[idx, 'event_date'] = corrupt_date(
            events.at[idx, 'event_date'], random.choice(['dmy', 'mdy']))
    print(f'DQ-ATS-02: date formats corrupted in ~8% of date fields')

    # DQ-ATS-03: Offer amounts as text strings (~12% of offers)
    def corrupt_amount(amount):
        style  = random.choice(['euro_symbol', 'k_suffix', 'european_decimal'])
        amount = int(amount)
        if style == 'euro_symbol':
            return f'EUR{amount:,}'
        elif style == 'k_suffix':
            return f'{amount // 1000}k'
        else:
            return f'{amount:,}'.replace(',', '.')

    corrupt_mask = offers.sample(frac=0.12, random_state=RANDOM_SEED + 3).index
    for idx in corrupt_mask:
        offers.at[idx, 'offer_amount'] = corrupt_amount(offers.at[idx, 'offer_amount'])
    print(f'DQ-ATS-03: {len(corrupt_mask)} offer amounts converted to text')

    # DQ-ATS-04: Duplicate candidate records (~3% of candidates)
    n_dupes   = int(len(cands) * 0.03)
    dupe_src  = cands.sample(n_dupes, random_state=RANDOM_SEED + 4)
    dupes     = []
    for _, row in dupe_src.iterrows():
        new_id = f'CAND-{len(cands) + len(dupes) + 1:04d}'
        first  = row['first_name']
        if len(first) > 3 and random.random() < 0.5:
            first = first[:-1]
        dupes.append({**row.to_dict(), 'candidate_id': new_id, 'first_name': first})
    cands = pd.concat([cands, pd.DataFrame(dupes)], ignore_index=True)
    print(f'DQ-ATS-04: {n_dupes} duplicate candidate records created')

    # DQ-ATS-05: Source field inconsistencies (~20% of applications)
    source_variants = {
        'LinkedIn Organic':    ['linkedin', 'Linked In', 'LI', 'LinkedIn', 'linkedin.com'],
        'LinkedIn Recruiter':  ['LinkedIn Recruiter', 'LI Recruiter', 'LinkedIn InMail'],
        'LinkedIn Paid':       ['LinkedIn Sponsored', 'LinkedIn Ads'],
        'Employee Referral':   ['Referral', 'referral', 'Employee Ref', 'Internal Referral'],
        'Indeed':              ['indeed', 'Indeed.com', 'INDEED'],
        'Glassdoor':           ['glassdoor', 'Glassdoor.com'],
        'Direct Application':  ['Direct', 'Careers Page', 'Company Website', 'Careers Site'],
        'Recruitment Agency':  ['Agency', 'agency', 'Headhunter', 'External Recruiter'],
        'GitHub Sourcing':     ['GitHub', 'github'],
    }

    dq05_mask = apps.sample(frac=0.20, random_state=RANDOM_SEED + 5).index
    for idx in dq05_mask:
        original = apps.at[idx, 'source']
        if original in source_variants:
            apps.at[idx, 'source'] = random.choice(source_variants[original])
    print(f'DQ-ATS-05: {len(dq05_mask)} source values made inconsistent')

    # DQ-ATS-06: Impossible date sequences (~2% of applications)
    dq06_apps = apps.sample(frac=0.02, random_state=RANDOM_SEED + 6)['application_id']
    dq06_mask = events[events['application_id'].isin(dq06_apps)].sample(
        frac=0.5, random_state=RANDOM_SEED + 7).index
    for idx in dq06_mask:
        try:
            d = date.fromisoformat(str(events.at[idx, 'event_date'])[:10])
            events.at[idx, 'event_date'] = to_iso(d - timedelta(days=random.randint(5, 20)))
        except:
            pass
    print(f'DQ-ATS-06: impossible date sequences in ~{len(dq06_apps)} applications')

    # DQ-ATS-07: Unclosed requisitions (~10% of filled jobs)
    filled_idx  = jobs[jobs['status'] == 'filled']
    n_unclosed  = max(1, int(len(filled_idx) * 0.10))
    unclosed    = filled_idx.sample(n_unclosed, random_state=RANDOM_SEED + 8).index
    for idx in unclosed:
        jobs.at[idx, 'status']                = 'open'
        jobs.at[idx, 'requisition_close_date'] = None
    print(f'DQ-ATS-07: {n_unclosed} filled jobs left with status = open')

    print('All DQ issues injected.')
    return jobs, cands, apps, events, offers


print('Injecting data quality issues...')
(jobs_raw, candidates_raw, applications_raw,
 pipeline_events_raw, offers_raw) = inject_dq_issues(
    jobs_df, candidates_df, applications_df, pipeline_events_df, offers_df)

Injecting data quality issues...
DQ-ATS-01: 2160 pipeline event rows affected
DQ-ATS-02: date formats corrupted in ~8% of date fields
DQ-ATS-03: 22 offer amounts converted to text
DQ-ATS-04: 126 duplicate candidate records created
DQ-ATS-05: 840 source values made inconsistent
DQ-ATS-06: impossible date sequences in ~84 applications
DQ-ATS-07: 2 filled jobs left with status = open
All DQ issues injected.


---
## Cell 14: SAVE ALL TABLES TO CSV

In [15]:
# =============================================================
# CELL 14: SAVE ALL TABLES TO CSV
# =============================================================
# Saves nine raw CSVs to Google Drive. These files are
# deliberately messy. The cleaning pipeline in Step 3 is
# what fixes them.
# =============================================================

def save_all(output_path: str):
    tables = {
        'talentflow_jobs':            jobs_raw,
        'talentflow_candidates':      candidates_raw,
        'talentflow_applications':    applications_raw,
        'talentflow_pipeline_events': pipeline_events_raw,
        'talentflow_offers':          offers_raw,
        'peoplecore_employees':       employees_df,
        'peoplecore_performance':     performance_df,
        'peoplecore_compensation':    compensation_df,
        'finance_headcount_plan':     headcount_plan_df,
    }

    for name, df in tables.items():
        path = f'{output_path}/{name}.csv'
        df.to_csv(path, index=False)
        print(f'  {name}.csv  ({len(df):,} rows)')

    print(f'\nAll files saved to: {output_path}')


save_all(PATHS['raw'])

  talentflow_jobs.csv  (45 rows)
  talentflow_candidates.csv  (4,326 rows)
  talentflow_applications.csv  (4,200 rows)
  talentflow_pipeline_events.csv  (14,399 rows)
  talentflow_offers.csv  (186 rows)
  peoplecore_employees.csv  (345 rows)
  peoplecore_performance.csv  (222 rows)
  peoplecore_compensation.csv  (120 rows)
  finance_headcount_plan.csv  (48 rows)

All files saved to: /content/drive/MyDrive/enova-talent-intelligence/data/raw


---
## Cell 15: VALIDATION

In [16]:
# =============================================================
# CELL 15: VALIDATION
# =============================================================
# Confirms the key statistics match expectations before
# moving to Step 3.
# =============================================================

print('=' * 55)
print('VALIDATION')
print('=' * 55)

print(f'\nVolume')
print(f'  Jobs:               {len(jobs_raw):>5}   (target: 45)')
print(f'  Candidates:         {len(candidates_raw):>5}   (target: ~4200+)')
print(f'  Applications:       {len(applications_raw):>5}   (target: ~4200)')
print(f'  Pipeline events:    {len(pipeline_events_raw):>5}')
print(f'  Offers:             {len(offers_raw):>5}   (target: ~200)')
print(f'  Employees:          {len(employees_df):>5}   (target: 345)')
print(f'  Performance:        {len(performance_df):>5}   (target: 280)')
print(f'  Comp changes:       {len(compensation_df):>5}   (target: 120)')
print(f'  Headcount plan:     {len(headcount_plan_df):>5}')

print(f'\nKey findings')
total_hires    = applications_raw['hired'].sum()
ref_apps       = applications_raw['source'].str.contains(
    'Referral|referral|EE Ref', na=False).sum()
ref_hires      = applications_raw[
    applications_raw['hired'] &
    applications_raw['source'].str.contains('Referral|referral', na=False)
].shape[0]
acc_rate       = (offers_raw['offer_status'] == 'accepted').mean()
active_count   = (employees_df['status'] == 'active').sum()
ats_link_rate  = employees_df['ats_candidate_id'].notna().mean()

print(f'  Total hires:             {total_hires}  (target: ~180)')
print(f'  Referral applications:   {ref_apps}  '
      f'({ref_apps/len(applications_raw):.1%}, target: ~11%)')
print(f'  Referral hires:          {ref_hires}  '
      f'({ref_hires/max(total_hires,1):.1%}, target: ~28%)')
print(f'  Offer acceptance rate:   {acc_rate:.1%}  (target: ~90%)')
print(f'  Active employees:        {active_count}  (target: 300)')
print(f'  ATS link rate:           {ats_link_rate:.1%}  (target: ~52%)')

rej_events      = pipeline_events_raw[pipeline_events_raw['event_type'] == 'rejected']
missing_reasons = rej_events['rejection_reason'].isna().mean()
non_canonical   = ~applications_raw['source'].isin(SOURCES)
print(f'  Missing rejection reasons: {missing_reasons:.1%}  (target: ~35%)')
print(f'  Non-canonical sources:     {non_canonical.mean():.1%}  (target: ~20%)')

print('\nStep 2 complete.')

VALIDATION

Volume
  Jobs:                  45   (target: 45)
  Candidates:          4326   (target: ~4200+)
  Applications:        4200   (target: ~4200)
  Pipeline events:    14399
  Offers:               186   (target: ~200)
  Employees:            345   (target: 345)
  Performance:          222   (target: 280)
  Comp changes:         120   (target: 120)
  Headcount plan:        48

Key findings
  Total hires:             166  (target: ~180)
  Referral applications:   455  (10.8%, target: ~11%)
  Referral hires:          45  (27.1%, target: ~28%)
  Offer acceptance rate:   89.2%  (target: ~90%)
  Active employees:        301  (target: 300)
  ATS link rate:           48.1%  (target: ~52%)
  Missing rejection reasons: 36.5%  (target: ~35%)
  Non-canonical sources:     17.8%  (target: ~20%)

Step 2 complete.
Next: run audit_and_clean.py (Step 3)
